# MagneticTurbulence — k-ε verification (PolyMAC_P0)

This form verifies the independent magnetic k-ε closure in two regimes:
frozen manufactured terms and a zero-production analytic decay. The
material uses ρ=μ₀=1, so the stored magnetic field equals b_A.


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()
BUILD = run.BUILD_DIRECTORY


## Frozen manufactured fields

The divergence-free velocity is u=(sin(2πx)cos(2πy), −cos(2πx)sin(2πy)).
The magnetic field is scalar/out-of-plane in 2D; its curl is therefore
−∂b_A/∂y. The transport equations are marked `equation_non_resolue` and
the time scheme has zero steps, matching TrioCFD's `Verification_RANS`
post-processing verification pattern. PolyMAC_MPFA does not implement
periodicity, so this first PolyMAC case uses homogeneous no-flux walls;
the analytic fields have zero normal derivatives there.


In [ ]:
NX = NY = 65
case = run.addCaseFromTemplate("magnetic_turbulence.data", "coarse",
                               dic={"nx": NX, "ny": NY},
                               nbProcs=1, excluNR=True)
mesh_levels = {"coarse": 17, "standard": 33, "fine": 65}
dt_levels = {"coarse": 0.125, "standard": 0.0625, "fine": 0.03125}
T_FINAL = 2.0
decay_cases = {}
for mesh_name, nodes in mesh_levels.items():
    for dt_name, dt in dt_levels.items():
        target = f"decay_{mesh_name}_{dt_name}"
        decay_cases[(mesh_name, dt_name)] = run.addCaseFromTemplate(
            "magnetic_turbulence_decay.data", target,
            dic={"nx": nodes, "ny": nodes, "dt": dt, "tmax": T_FINAL},
            excluNR=True)
run.printCases()
run.runCases()
print("Frozen manufactured case completed.")


## Zero-production analytic decay

A uniform b_A has zero curl, so P_b=0; uniform k_b and ε_b with zero
velocity have no convective or diffusive transport. The remaining ODEs are
dk_b/dt=−ε_b and dε_b/dt=−C₂ε ε_b²/k_b. With r₀=ε₀/k₀, their exact
solution is k_b=k₀ f(t)^(−1/(C₂ε−1)) and
ε_b=ε₀ f(t)^(−C₂ε/(C₂ε−1)), where f(t)=1+(C₂ε−1)r₀t. Three mesh levels are crossed with three time steps;
the uniform state should be mesh-independent while time error decreases.


In [ ]:
import pathlib
import numpy as np

C2_EPS = 1.92
K0, EPS0 = 1.e-6, 1.e-7
decay_results = {}
decay_states = {}
build_dir = pathlib.Path(BUILD)

def analytic_decay(time):
    factor = 1.0 + (C2_EPS-1.0)*(EPS0/K0)*time
    return (K0*factor**(-1.0/(C2_EPS-1.0)),
            EPS0*factor**(-C2_EPS/(C2_EPS-1.0)))

for (mesh_name, dt_name), decay_case in decay_cases.items():
    values = {}
    for field in ("K", "EPSILON"):
        probe = build_dir / decay_case.dir_ / (
            f"{decay_case.dataFileName_}_SONDE_{field}.son")
        samples = np.atleast_2d(np.loadtxt(probe, comments="#"))
        time, value = samples[:, 0], samples[:, 1]
        assert np.all(np.isfinite(value)), f"non-finite {field} in {probe}"
        assert np.min(value) > 0.0, f"{field} lost positivity in {probe}"
        assert abs(time[-1] - T_FINAL) < 1.e-8, (
            f"{probe} ended at t={time[-1]}, expected {T_FINAL}")
        exact_k, exact_eps = analytic_decay(time[-1])
        exact = exact_k if field == "K" else exact_eps
        scale = K0 if field == "K" else EPS0
        values[field] = value[-1]
        decay_results[(mesh_name, dt_name, field)] = abs(value[-1]-exact)/scale
        decay_states[(mesh_name, dt_name, field)] = value[-1]
    for term in ("CONV_K", "DIFF_K", "CONV_EPSILON", "DIFF_EPSILON"):
        probe = build_dir / decay_case.dir_ / (
            f"{decay_case.dataFileName_}_SONDE_{term}.son")
        samples = np.atleast_2d(np.loadtxt(probe, comments="#"))
        term_max = float(np.max(np.abs(samples[:, 1])))
        assert term_max < 1.e-12, (
            f"{term} should vanish for uniform fields/zero velocity, got {term_max:.3e}")
    ek = decay_results[(mesh_name, dt_name, "K")]
    ee = decay_results[(mesh_name, dt_name, "EPSILON")]
    print(f"{mesh_name:8s} dt={dt_levels[dt_name]:.5f}  "
          f"k={values['K']:.8e} (rel.err {ek:.3e})  "
          f"epsilon={values['EPSILON']:.8e} (rel.err {ee:.3e})")

for mesh_name in mesh_levels:
    for field in ("K", "EPSILON"):
        errors = [decay_results[(mesh_name, dt_name, field)]
                  for dt_name in dt_levels]
        assert errors[2] < errors[1] < errors[0], (
            f"{field} error did not decrease with dt on {mesh_name}: {errors}")

for dt_name in dt_levels:
    for field in ("K", "EPSILON"):
        mesh_values = [decay_states[(mesh_name, dt_name, field)]
                       for mesh_name in mesh_levels]
        assert max(mesh_values)-min(mesh_values) < 1.e-8, (
            f"uniform {field} decay depends on mesh at dt={dt_name}: {mesh_values}")

print("Zero-production analytic decay: PASS (positive, time-convergent, mesh-independent).")


## Analytic reference and term checks

The reference uses explicitly differentiated analytic expressions for
∇·(D∇q), where D includes molecular plus
turbulent diffusivity. TRUST's convection operator is assembled on the
right-hand side and is compared with −u·∇q. The source references include
P_b=η_t|∇×b_A|² and the implemented k_b/ε_b source formulas.


In [ ]:
import pathlib
import numpy as np
from trustutils import plot

x_probe = 0.2578125
Cmu, Ce1, Ce2 = 0.09, 1.44, 1.92
sigma_k, sigma_eps, Pm_t, eta = 1.0, 1.3, 1.0, 0.01
twopi = 2.0*np.pi

def analytic_terms(y_coord):
    xx = np.full_like(y_coord, x_probe)
    ax, ay = twopi*xx, twopi*y_coord
    sx, cx, sy, cy = np.sin(ax), np.cos(ax), np.sin(ay), np.cos(ay)
    ux, uy = sx*cy, -cx*sy
    k = 2.0 + 0.25*cx + 0.15*cy
    eps = 3.0 + 0.2*cx + 0.1*cy
    kx, ky = -0.5*np.pi*sx, -0.3*np.pi*sy
    ex, ey = -0.4*np.pi*sx, -0.2*np.pi*sy
    kxx, kyy = -(np.pi**2)*cx, -0.6*(np.pi**2)*cy
    exx, eyy = -0.8*(np.pi**2)*cx, -0.4*(np.pi**2)*cy
    eta_t = Cmu*k*k/(eps*Pm_t)
    etax = eta_t*(2.0*kx/k - ex/eps)
    etay = eta_t*(2.0*ky/k - ey/eps)
    production = eta_t*(twopi*sy)**2  # |curl(b_A)|^2, field_scale=1
    conv_k = -(ux*kx + uy*ky)
    conv_eps = -(ux*ex + uy*ey)
    d_k, d_eps = eta + eta_t/sigma_k, eta + eta_t/sigma_eps
    dx_k, dy_k = etax/sigma_k, etay/sigma_k
    dx_eps, dy_eps = etax/sigma_eps, etay/sigma_eps
    diff_k = d_k*(kxx+kyy) + dx_k*kx + dy_k*ky
    diff_eps = d_eps*(exx+eyy) + dx_eps*ex + dy_eps*ey
    source_k = production - eps
    source_eps = Ce1*(eps/k)*production - Ce2*eps*eps/k
    return {
        "CONV_K": conv_k, "DIFF_K": diff_k, "SOURCE_K": source_k,
        "CONV_EPSILON": conv_eps, "DIFF_EPSILON": diff_eps,
        "SOURCE_EPSILON": source_eps,
    }

references = {
    "CONV_K": 0.12, "DIFF_K": 0.12, "SOURCE_K": 0.03,
    "CONV_EPSILON": 0.12, "DIFF_EPSILON": 0.12,
    "SOURCE_EPSILON": 0.03,
}

build_dir = pathlib.Path(BUILD)
data_base = f"coarse/{case.dataFileName_}"
results = {}
for tag, tolerance in references.items():
    run.extraitCoupe(data_base, tag, directory=str(build_dir))
    coupe = build_dir / "coarse" / f"{case.dataFileName_}_{tag}.coupe"
    columns = plot.loadText(str(coupe))
    coordinate = np.asarray(columns[0], dtype=float)
    actual = np.asarray(columns[1], dtype=float)
    expected = analytic_terms(coordinate)[tag]
    error = float(np.max(np.abs(actual - expected)))
    scale = max(float(np.max(np.abs(expected))), 1.e-12)
    relative_error = error / scale
    assert relative_error < tolerance, (
        f"{tag}: relative max error {relative_error:.3e} >= {tolerance:.3e}")
    results[tag] = (relative_error, tolerance)
    print(f"{tag:14s} relative max error = {relative_error:.3e} (limit {tolerance:.3e})")

print("Manufactured-term verification: PASS")
run.tablePerf()
